# Inference and Information Extraction

This notebook implements the **inference stage** of the Emergency Care Disruption Dataset (ECDD) pipeline.

In the previous stage of the pipeline, a web retrieval process was used to collect documents related to French hospital emergency departments using the Tavily Search API. Each hospital in the FINESS registry was used to generate targeted queries, producing a set of candidate sources (news articles, institutional communications, and press releases).

The objective of this notebook is to **analyze the retrieved documents and extract structured information about emergency department disruptions**.

Using a local large language model (**Mistral-7B-Instruct**, deployed with **vLLM**), the pipeline processes the textual content of each source and identifies relevant events such as:

- temporary emergency department closures  
- regulated access to emergency services  
- strikes affecting emergency care  
- SMUR service disruptions  

The extracted information is then converted into structured records that can be stored in a long-format dataset where **each row corresponds to a single event supported by a verifiable source URL**.

Within the overall pipeline, this notebook corresponds to the **inference** stage of the pipeline, transforming unstructured web content into machine-readable data for subsequent structuring and analysis.

In [45]:
import json
import time
import os
import re
import pandas as pd
from datetime import datetime
from vllm import LLM, SamplingParams

ideas:
1) https://www.youtube.com/watch?app=desktop&v=9j-480mlXEk&start=0

## Local LLM Setup: Why Mistral-7B and Why vLLM?

This notebook uses a **local large language model**, `Mistral-7B-Instruct-v0.3`, served through **vLLM**, to perform structured information extraction from the retrieved web sources. 

### Why use a local model?

A local model was preferred over an external API for three main reasons:

1. **Reproducibility**  
   Running the model locally makes the extraction pipeline easier to reproduce, since the same model version and inference settings can be reused across runs.

2. **Auditability and control**  
   The inference process remains fully controlled within the project environment. This is particularly important when processing hospital-level evidence and preserving a transparent extraction pipeline.

3. **Scalability for batch processing**  
   The task requires repeated inference over many retrieved documents. A local setup allows large batches of extractions without depending on API quotas or external service availability.

### Why `Mistral-7B-Instruct-v0.3`?

[`Mistral-7B-Instruct-v0.3`](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3) was selected as a practical compromise between **instruction-following ability**, **computational efficiency**, and **local deployability**.


This model is suitable for the current task because:

- it can follow structured prompts reliably,
- it is lightweight enough to run on local GPU infrastructure,
- it performs well for extraction-style tasks where the goal is to return concise, schema-constrained outputs rather than long-form generation.

In this notebook, the model is not used for open-ended text generation, but as a **controlled extractor** of event attributes from retrieved sources.

### Why use vLLM?

The model is served through [`vLLM`](https://github.com/vllm-project/vllm), an inference engine designed for efficient deployment of large language models.

vLLM is used here because it provides:

- **fast inference** for repeated prompt execution,
- **efficient GPU memory management**,
- a simple interface for running the model locally in batch settings.

This makes it well suited for a pipeline in which many documents must be processed sequentially or in batches.

### Inference settings

The model is loaded from disk and configured with the following parameters:

- `dtype='half'`  
  Uses half-precision floating point format to reduce GPU memory usage.

- `gpu_memory_utilization=0.5`  
  Limits the fraction of GPU memory allocated to the model, helping avoid memory saturation.

- `max_model_len=2000`  
  Restricts the maximum input context length, which is sufficient for the extraction prompts used in this notebook while remaining computationally manageable.

- `temperature=0`  
  Forces deterministic generation, which is appropriate for structured extraction tasks where consistency is preferred over creativity.

- `max_tokens=256`  
  Caps the output length, since the expected output is a short structured response rather than free-form text.



In [52]:
# Force vLLM to use the Data partition for everything
os.environ['VLLM_CONFIG_ROOT'] = '/Data/anahi_reyes/vllm_cache/config'
os.environ['VLLM_CACHE_ROOT'] = '/Data/anahi_reyes/vllm_cache'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['HF_HOME'] = '/Data/anahi_reyes/huggingface_cache'
# Forzamos el uso del motor estable (V0)
#os.environ["VLLM_USE_V1"] = "false"
# También por seguridad, evitamos que intente usar HTTP moderno si hay conflictos de red local
#os.environ["VLLM_CONFIGURE_LOGGING"] = "1"

# Load the in-disk LLM to the environment
model_path = '/Data/anahi_reyes/models/Mistral-7B-Instruct-v0.3'
llm = LLM(model=model_path, dtype='half', gpu_memory_utilization=0.8, max_model_len=4096)
sampling_params = SamplingParams(temperature=0, max_tokens=512,  stop=["</json>", "[/INST]"])


INFO 03-14 13:37:14 [utils.py:261] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': '/Data/anahi_reyes/models/Mistral-7B-Instruct-v0.3'}
INFO 03-14 13:37:14 [model.py:541] Resolved architecture: MistralForCausalLM
WARNING 03-14 13:37:14 [model.py:1885] Casting torch.bfloat16 to torch.float16.
INFO 03-14 13:37:14 [model.py:1561] Using max model len 4096
INFO 03-14 13:37:14 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=876260) INFO 03-14 13:37:14 [core.py:96] Initializing a V1 LLM engine (v0.15.0) with config: model='/Data/anahi_reyes/models/Mistral-7B-Instruct-v0.3', speculative_config=None, tokenizer='/Data/anahi_reyes/models/Mistral-7B-Instruct-v0.3', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_siz

(EngineCore_DP0 pid=876260) Process EngineCore_DP0:
(EngineCore_DP0 pid=876260) Traceback (most recent call last):
(EngineCore_DP0 pid=876260)   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore_DP0 pid=876260)     self.run()
(EngineCore_DP0 pid=876260)   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/multiprocessing/process.py", line 108, in run
(EngineCore_DP0 pid=876260)     self._target(*self._args, **self._kwargs)
(EngineCore_DP0 pid=876260)   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 950, in run_engine_core
(EngineCore_DP0 pid=876260)     raise e
(EngineCore_DP0 pid=876260)   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 937, in run_engine_core
(EngineCore_DP0 pid=876260)  

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [53]:
#df = df = pd.read_json("/Data/anahi_reyes/EDCD_data/raw_edcd_database_atomic.jsonl", lines=True)
df = pd.read_json("https://raw.githubusercontent.com/AnahiRM/PiA-DREES-2025/main/data/intermediate/raw_edcd_database_atomic.jsonl", lines=True)
df.head()

,finess,hospital_name,nom_etab_long,keywords_nom_etab,source_url,title,content,score,retrieved_at
0,010000024,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,BOURG-EN-BRESSE FLEYRIAT,https://clinique-convert-bourg-en-bresse.ramsaysante.fr/actualites/fermeture-temporaire-des-urgences,Fermeture temporaire des Urgences - Clinique Convert,"Avant de venir\n\n + Conseils pratiques de A à Z\n Plan d'accès\n\n Rechercher\n\n# Actualités\n\n## Fermeture temporaire des Urgences\n\nle 11/07/2025\n\nRetour\n\nLe service des urgences de notre établissement fermera temporairement l’accueil de nuit du vendredi 11 juillet à 17h00 au mercredi 16 juillet à 8h00.\n\nDurant cette période, les urgences seront ouvertes chaque jour de 8h00 à 17h00.\n\nLes urgences cardiologiques restent assurées.\n\nPendant cette période, Il est demandé aux patients de se présenter au centre hospitalier de Fleyriat. Celui-ci recevra le renfort des médecins urgentistes de la Clinique Ramsay Convert. [...] - Préparez votre séjour\n - Votre sortie\n + Droits du patient\n\n - Vos droits et devoirs\n - Les représentants des usagers\n - Votre dossier médical\n - Lettre du pharmacien au patient : information sur votre prise en charge médicamenteuse\n Trouver un médecin\n Vous êtes accompagnant\n\n + Se rendre facilement à la clinique Convert\n\n - Comment venir à la clinique Convert ?\n - Les visites\n + Sur place\n Informations pratiques\n\n + Avant de venir\n\n - Conseils pratiques de A à Z\n + Plan d'accès\n\nMon compte Ramsay Services\n\n Bienvenue à la Clinique Convert\n Découvrez la clinique Convert\n\n + La clinique Convert : un établissement de soins pluridisciplinaire de proximité\n Nos compétences\n NOTRE PHILOSOPHIE [...] Aller au contenu principal \n\nRamsay Santé Clinique Convert\n\nMon compte Ramsay Services\n\n Présentation établissement\n\n + Bienvenue à la Clinique Convert\n + Découvrez la clinique Convert\n\n - La clinique Convert : un établissement de soins pluridisciplinaire de proximité\n + Nos compétences\n + NOTRE PHILOSOPHIE\n\n - Le suivi patient\n - La certification de la Haute Autorité de Santé\n + Qualité / gestion des risques\n\n - La Démarche Qualité\n - La prévention des risques\n Vous êtes patient\n\n + Pourquoi choisir Ramsay Générale de Santé ?\n\n - Nos prises en charge\n - Tous nos soins\n - Nos Services\n + Votre hospitalisation\n\n - Hospitalisation en chirurgie\n - Hospitalisation en médecine\n - Vos frais d'hospitalisation\n + Votre séjour à la clinique Convert",0.672182,2026-03-14 11:50:40
1,010000024,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,BOURG-EN-BRESSE FLEYRIAT,https://www.leprogres.fr/sante/2025/10/16/urgences-de-l-hopital-de-bourg-en-bresse-le-retour-a-la-normale-annonce-apres-dix-mois-de-restrictions,Ain. Urgences de l'hôpital de Bourg-en-Bresse - Le Progrès,"Le 31 janvier dernier, les urgences du centre hospitalier de Bourg-en-Bresse (CHB) étaient réorganisées du vendredi soir et jusqu’au dimanche matin. De 17 heures à 8 heures le lendemain, le service tournait en effectif réduit avec deux médecins urgentistes en poste, dont celui du Samu, contre quatre en temps normal. Cette décision avait été prise après une longue réflexion de l’établissement qui constatait que les professionnels étaient usés du rythme imposé. Cela devait permettre à l’équipe de souffler un peu et de ne plus travailler trois week-ends sur quatre dans un contexte d’augmentation de 7 % des passages aux urgences en 2024. Chaque année, 52 000 personnes franchissent la porte de ce service. [...] Ce fonctionnement temporaire est désormais terminé ou presque. Le retour à la normale est annoncé pour le 7 novembre. « C’est une bonne nouvelle, confirme Frédérique Labro-Gouby, la...\n\n...pour lire la suite, rejoignez notre communauté d'abonnés\n\net accédez à l'intégralité de nos articles sur le site et l'application mobile\n\nà partir de 1 € le 1er mois, sans engagement de durée\n\n{'skus': ['lprswgpremium16']}\n\nGoogle : 1€ le

In [54]:
def build_extraction_prompt(row):
    h_name = row['hospital_name']
    h_keywords = row['keywords_nom_etab']
    return f"""<s>[INST]
You are a structured data extraction assistant specializing in French healthcare disruptions.

TARGET HOSPITAL:
- Official name: "{h_name}"
- Also known as: "{h_keywords}"
- Match even with abbreviations or partial names.
- Do NOT match other hospitals sharing only a generic word like "CHU" or "Centre".

TASK:
Determine whether THIS TARGET HOSPITAL's emergency department (Urgences) is experiencing
a confirmed disruption, a potential disruption, or neither.

Set relevant = true if the article explicitly states that the urgences are:
- Closed (fully or partially, day or night)
- Operating with reduced staffing or reorganized access
- Regulated (patients must call before coming)
- Redirecting ambulances
- The article describes a past disruption that has now ended — the event still occurred and should be captured.

Set relevant = "potential" if no disruption has been implemented yet, but the article
explicitly describes a direct threat to the urgences:
- Documented staff shortage or upcoming reorganization
- ARS intervention or administrative crisis specifically affecting the urgences
- Political motions or demands specifically about the urgences

Set relevant = false in all other cases, including:
- General hospital tensions without specific urgences impact
- Disruptions in other departments (surgery, maternity, psychiatry, etc.)
- The hospital appears only as a backup/receiving hospital
- Proposals or debates with nothing implemented yet

When in doubt, prefer false over true or potential.

ROLE IDENTIFICATION:
STEP 1: Copy the exact name of the hospital whose urgences are closing, regulated, or at risk.
STEP 2: Does that name contain "{h_name}" or "{h_keywords}"?
        - Yes → proceed to assess relevant = true or "potential"
        - No → relevant = false, regardless of whether the target hospital is mentioned elsewhere.
STEP 3: If the TARGET HOSPITAL is mentioned only as receiving patients, it is OPEN → relevant = false.

TEMPORAL REASONING:
- The publication date may appear in the URL (e.g. /2024/03/15/ or /2024-03-15/).
- If only month and year are known, use the first of the month: "2022-07-01".
- If found in the URL, use it as the publication date.
- If also present in the article text, prefer the article text date.
- WARNING: ignore dates found in unrelated links, footers, or "related articles" sections.
- Use it to resolve relative expressions: "today", "this weekend", "from Monday".
- If no publication date is found, leave relative expressions as "Unknown" — do not guess.
- end_date = "Ongoing" only if the article explicitly says no end date is set.
- end_date = "Unknown" if a closure is mentioned but no end date is given.
- end_date = "YYYY-MM-DD" if a specific date is stated or clearly implied.

OUTPUT:
First write 1-2 sentences explaining your decision.
Then output the result in a <json> block.

Confirmed disruption case:
<json>
{{
  "relevant": true,
  "status": "choose one: Fermeture complète / Fermeture temporaire / Régulation des entrées / Détournement ambulances / Unknown",
  "start_date": "YYYY-MM-DD | Unknown",
  "end_date": "YYYY-MM-DD | Ongoing | Unknown",
  "reason": "Free text — describe the cause of the disruption as mentioned. Unknown if not mentioned.",
  "publication_date": "YYYY-MM-DD | Unknown",
  "confidence": "High | Medium | Low"
}}
</json>

Potential disruption case:
<json>
{{
  "relevant": "potential",
  "risk_description": "Free text — describe what situation puts the urgences at risk.",
  "publication_date": "YYYY-MM-DD | Unknown",
  "confidence": "High | Medium | Low"
}}
</json>

Non-relevant case:
<json>{{"relevant": false}}</json>

TEXT:
URL: {row['source_url']}
Title: {row['title']}
Content: {row['content']}
[/INST]"""

#def hospital_mention_score(row):
    content = str(row['content']).lower()
    title = str(row['title']).lower()
    name = str(row['hospital_name']).lower()
    keywords = str(row['keywords_nom_etab']).lower()
    nom_long = str(row['nom_etab_long']).lower()
    
    generic = {'de', 'du', 'le', 'la', 'les', 'en', 'et', 'ch',
               'centre', 'hospitalier', 'clinique', 'hopital', 'none',
               'chu', 'ght', 'ghm', 'chi', 'chl'}
    
    all_names = name + ' ' + keywords + ' ' + nom_long
    target_words = {w for w in set(all_names.split()) - generic if len(w) > 4}
    
    text = content + ' ' + title
    matches = sum(1 for word in target_words if word in text)
    return matches

def robust_json_parse(raw_output):
    match = re.search(r'<json>(.*?)</json>', raw_output, re.DOTALL)
    if not match:
        match = re.search(r'(\{.*\})', raw_output, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1).strip())
        except json.JSONDecodeError as e:
            return {"relevant": False, "parsing_error": True, "error_msg": str(e), "raw": raw_output[:200]}
    return {"relevant": False, "no_json_found": True, "raw": raw_output[:200]}

# Execution
#prompts = df.apply(build_extraction_prompt, axis=1).tolist()
#responses = llm.generate(prompts, sampling_params)

#df_ext = pd.DataFrame([robust_json_parse(r.outputs[0].text) for r in responses])

In [55]:
# ── Load data ──────────────────────────────────────────────────────────────────
df = pd.read_json(
    "https://raw.githubusercontent.com/AnahiRM/PiA-DREES-2025/main/data/intermediate/raw_edcd_database_atomic.jsonl",
    lines=True
)
print(f"Total rows loaded: {len(df)}")

# ── Pre-filter: only rows where target hospital is mentioned in content/title ──
# NOTE: Current limitation — uses exact keyword matching on words > 4 chars.
# This may miss informal references on social media (e.g. "l'hosto de Brive").
# Proposed next step: replace with fuzzy matching or city-name extraction.
#def mentions_target_hospital(row):
#    content = str(row['content']).lower()
#    title = str(row['title']).lower()
#    name = str(row['hospital_name']).lower()
#    keywords = str(row['keywords_nom_etab']).lower()
#    nom_long = str(row.get('nom_etab_long', '')).lower()

#    generic = {'de', 'du', 'le', 'la', 'les', 'en', 'et', 'ch',
#               'centre', 'hospitalier', 'clinique', 'hopital', 'none',
#               'chu', 'ght', 'ghm', 'chi', 'chl'}

#    all_names = name + ' ' + keywords + ' ' + nom_long
#    target_words = {w for w in set(all_names.split()) - generic if len(w) > 4}

#    text = content + ' ' + title
#    return any(word in text for word in target_words)

#df_filtered = df[df.apply(mentions_target_hospital, axis=1)].reset_index(drop=True)
#print(f"Rows after pre-filter: {len(df_filtered)} / {len(df)}")

# ── Truncate content to avoid exceeding max_model_len ─────────────────────────
#df_filtered = df_filtered.copy()
#df_filtered['content'] = df_filtered['content'].str[:8000]

# ── Checkpointing: skip already processed rows ────────────────────────────────
output_path = "/Data/anahi_reyes/EDCD_data/llm_output.jsonl"

already_processed = set()
try:
    df_existing = pd.read_json(output_path, lines=True)
    already_processed = set(df_existing['source_url'].tolist())
    print(f"Resuming — {len(already_processed)} rows already processed")
except Exception:
    print("No existing output found, starting fresh")

df_to_process = df[~df['source_url'].isin(already_processed)].reset_index(drop=True)
print(f"Rows to process: {len(df_to_process)}")

# ── LLM inference ──────────────────────────────────────────────────────────────
prompts = df_to_process.apply(build_extraction_prompt, axis=1).tolist()
responses = llm.generate(prompts, sampling_params)

# ── Parse and save incrementally ──────────────────────────────────────────────
results = []
for i, r in enumerate(responses):
    parsed = robust_json_parse(r.outputs[0].text)
    entry = {**df_to_process.iloc[i].to_dict(), **parsed}
    results.append(entry)

    # Save every 100 rows as checkpoint
    if (i + 1) % 100 == 0:
        pd.DataFrame(results).to_json(output_path, orient='records', lines=True, mode='a', force_ascii=False)
        results = []
        print(f"Checkpoint saved at row {i + 1}")

# Save remaining
if results:
    pd.DataFrame(results).to_json(output_path, orient='records', lines=True, mode='a', force_ascii=False)

print(f"Done. Output saved to {output_path}")

# ── Build final dataframe ──────────────────────────────────────────────────────
df_final = pd.read_json(output_path, lines=True)
print(f"Total rows in output: {len(df_final)}")
print(f"Relevant disruptions: {df_final['relevant'].value_counts().to_dict()}")

Total rows loaded: 6072
No existing output found, starting fresh
Rows to process: 6072


/tmp/ipykernel_812603/3100600377.py:41: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_existing = pd.read_json(output_path, lines=True)


Adding requests:   0%|          | 0/6072 [00:00<?, ?it/s]

/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/site-packages/mistral_common/tokens/tokenizers/sentencepiece.py:133: FutureWarning: `get_control_token` is deprecated. Use `get_special_token` instead.
  warnings.warn("`get_control_token` is deprecated. Use `get_special_token` instead.", FutureWarning)


Processed prompts:   0%|          | 0/6072 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

Checkpoint saved at row 100
Checkpoint saved at row 200
Checkpoint saved at row 300
Checkpoint saved at row 400
Checkpoint saved at row 500
Checkpoint saved at row 600
Checkpoint saved at row 700
Checkpoint saved at row 800
Checkpoint saved at row 900
Checkpoint saved at row 1000
Checkpoint saved at row 1100
Checkpoint saved at row 1200
Checkpoint saved at row 1300
Checkpoint saved at row 1400
Checkpoint saved at row 1500
Checkpoint saved at row 1600
Checkpoint saved at row 1700
Checkpoint saved at row 1800
Checkpoint saved at row 1900
Checkpoint saved at row 2000
Checkpoint saved at row 2100
Checkpoint saved at row 2200
Checkpoint saved at row 2300
Checkpoint saved at row 2400
Checkpoint saved at row 2500
Checkpoint saved at row 2600
Checkpoint saved at row 2700
Checkpoint saved at row 2800
Checkpoint saved at row 2900
Checkpoint saved at row 3000
Checkpoint saved at row 3100
Checkpoint saved at row 3200
Checkpoint saved at row 3300
Checkpoint saved at row 3400
Checkpoint saved at row

In [43]:
df_ext.tail()

,relevant,publication_date,status,start_date,end_date,reason,confidence,risk_description
25,True,2022-08-01,Fermeture complète,2022-07-01,Ongoing,Nightly closure due to staff shortage,High,NaN
26,True,2022-05-18,Régulation des entrées,2022-05-18,Ongoing,Lack of staff,High,NaN
27,True,2025-12-01,Fermeture complète,2024-02-01,Ongoing,Night closure and daytime access only on the 15th of each month,High,NaN
28,potential,2025-07-24,NaN,NaN,NaN,NaN,NaN,"The article mentions that the 15 (regulation by the 15) has been generalized in Rennes, a practice that was first implemented in Digne-les-Bains and Manosque. This suggests a potential disruption in the urgences of the CH de Rambouillet in Rennes."
29,True,2023-07-01,Régulation des entrées,2023-07-01,Ongoing,The emergency department is reserved for the most serious cases due to tensions on the urgences and the hospital offer in the context of the cyberattack on the Centre Hospitalier de Versailles.,High,NaN


In [ ]:
df_final = pd.concat([df.reset_index(drop=True), df_ext.reset_index(drop=True)], axis=1)
#df_final = pd.concat([df.reset_index(drop=True), df_ext.reset_index(drop=True)], axis=1)
#pd.set_option('display.max_colwidth', None)
df_final.tail()

,finess,hospital_name,nom_etab_long,keywords_nom_etab,source_url,title,content,score,retrieved_at,relevant,publication_date,status,start_date,end_date,reason,confidence,risk_description
25,780000329,CH DE RAMBOUILLET,CENTRE HOSPITALIER DE RAMBOUILLET,RAMBOUILLET,https://www.samu-urgences-de-france.fr/medias/files/sudf_enquete_202207_resultats_VF.pdf,[PDF] RÉSULTATS DE L'ENQUÊTE SUdF SITUATION DES URGENCES ...,"totale de leur UHCD. Les SU ont enregistré durant le mois de juillet 2022 une augmentation d'activité en moyenne de 12,3% soit environ 180.000 passages de plus qu'en 2021 sur la même période. Cette augmentation est de 10% dans les départements où une régulation médicale préalable à l’accès au SU a été mise en place (cf. carte d’augmentation de l’activité des SU). 88 établissements (26%) ont mis en place une restriction d’accès, dont 67 avec une régulation médicale systématique par le Samu-Centre 15 pour autoriser l’accès aux urgences (recommandation n°23). 42 établissements ont été contraints de réaliser une fermeture totale de nuit de leur SU pour un nombre cumulé de 546 nuits en juillet. De jour ce sont 23 établissements qui ont réalisé une fermeture totale pour un nombre cumulé de 208 [...] (sur les 102 du territoire national). Concernant leurs ressources humaines, ils déclarent être en difficulté sur les ressources médicales pour 98% d’entre eux et pour les ressources non médicales pour 95%. 68% de ces établissements ont recours à des solutions d’intérim durant cet été. Les SAMU ont enregistré durant le mois de juillet 2022 une augmentation d'activité en moyenne de 21,5% comparativement à celle de 2021 à la même période. 21 départements ont une augmentation supérieure à 30%, dont 42% ayant mis en place une régulation médicale systématique pour autoriser l’accès aux urgences (cf. carte d’augmentation de l’activité des SAMU). Au total 42 départements (43%) ont mis en œuvre une régulation médicale systématique par le SAMU-Centre 15 avant l’accès aux SU (recommandation n°23). [...] n°27). SUdF - Situation des Urgences Juillet 2022 5 Commentaires : Compte tenu de la suractivité inhabituelle s’ajoutant aux flux estivaux importants dans certaines régions, du déficit majeur de personnels soignants et médicaux, et de l’indisponibilité des lits, très insuffisants pour permettre les hospitalisations nécessaires, les SU sont en très grande fragilité. La mise en œuvre des recommandations de la mission flash est insuffisante et ne permet pas d’assurer une fluidité et un fonctionnement sécuritaire dans le SU. La situation attendue au mois août va encore se dégrader, avec une augmentation des fermetures institutionnelles de lits et une diminution de la disponibilité de l’offre soignante libérale liée aux congés. Sans mesures contraignantes, il faut s’attendre à l’évolution",0.995245,2026-03-14 11:31:23,True,2022-08-01,Fermeture complète,2022-07-01,Ongoing,Nightly closure due to staff shortage,High,NaN
26,780000329,CH DE RAMBOUILLET,CENTRE HOSPITALIER DE RAMBOUILLET,RAMBOUILLET,https://www.franceinfo.fr/economie/greve/greve-aux-urgences/hopital-faute-de-soignants-des-services-durgences-ferment-la-nuit_5145202.html,"Hôpital : faute de soignants, des services d'urgences ferment la nuit","Dès mercredi 18 mai, il sera impossible pour les Bordelais de se rendre dans cet hôpital, entre 20h et 8h du matin, sauf accord préalable du Samu. Le plus important centre hospitalier de la Gironde étouffe. La fréquentation a augmenté de près de 50 % depuis le début de la crise sanitaire​, mais de nombreux médecins urgentistes ont quitté leur poste. Ce système de régulation pourrait réduire le nombre d’entrées d’une trentaine de patients chaque jour​, et pourrait s’inscrire dans la durée. \n\n## Un problème à l’échelle nationale [...] ## Un problème à l’échelle nationale\n\nLa situation est similaire dans le Vaucluse, où les urgences fermeront la nuit, faute de personnels. Les habitants sont inquiets. Selon les syndicats, 66 services d’urgence seraient